# KA-4: retrieval, cited local composition and layered evaluation

Read Chapters 44–46. This is an actual CPU pipeline using immutable KA-0 cards, an authored dense map and saved learned NNLM vectors. It does not train a new encoder or call an LLM. The generator is a deterministic exact-source composer. Code: Apache-2.0; explanatory text and fictional data: CC BY-SA 4.0.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
assert (ROOT / 'src/config/book.mjs').exists(), 'Run from the book repository root'
sys.path.insert(0, str(ROOT / 'code/knowledge-assistant'))
from contracts import load, encode, build_context, validate_output, provider_boundary


## Independent arithmetic before the general implementation
The five card lengths are 3, 4, 4, 2, 2. Beijing occurs in three documents; lodging in two. Decimal arithmetic below follows the hand derivation, separately from the BM25 implementation.

In [2]:
from decimal import Decimal, localcontext
from retrieval import run as retrieve, metrics
with localcontext() as ctx:
    ctx.prec = 40
    beijing = (Decimal(12) / 7).ln()
    lodging = (Decimal(12) / 5).ln()
    b_score = beijing * Decimal('0.88') + lodging * Decimal(44) / 35
ranking = retrieve()
actual = ranking['locales']['en']['rows'][0]['scores']['bm25']['travel-v2']
assert abs(actual - float(b_score)) < 1e-12
print('independent B score:', float(b_score))
for locale, result in ranking['locales'].items():
    print(locale, result['inherited']['success_counts'], result['success_counts'])
    for row in result['rows']:
        print(row['query_id'], row['returned'])


independent B score: 1.574906190461096
en {'keyword': 5, 'count': 6, 'tfidf': 7} {'bm25': 7, 'dense_authored': 4, 'dense_nnlm': 7, 'hybrid': 4, 'reranked': 8}
current-lodging {'bm25': ['travel-v2', 'travel-v1', 'rail-faq-v1'], 'dense_authored': ['travel-v1', 'travel-v2', 'rail-faq-v1'], 'dense_nnlm': ['travel-v2', 'travel-v1', 'rail-faq-v1'], 'hybrid': ['travel-v1', 'travel-v2', 'rail-faq-v1'], 'reranked': ['travel-v1', 'travel-v2', 'rail-faq-v1']}
geographic-status {'bm25': ['status-faq-v1', 'rail-faq-v1', 'travel-v1', 'travel-v2'], 'dense_authored': ['travel-v1', 'travel-v2', 'status-faq-v1', 'approval-faq-v1', 'rail-faq-v1'], 'dense_nnlm': ['status-faq-v1', 'approval-faq-v1', 'rail-faq-v1', 'travel-v2'], 'hybrid': ['travel-v1', 'status-faq-v1', 'travel-v2', 'rail-faq-v1', 'approval-faq-v1'], 'reranked': ['status-faq-v1', 'travel-v1', 'travel-v2']}
explicit-current {'bm25': ['travel-v2', 'travel-v1'], 'dense_authored': ['travel-v1', 'travel-v2', 'rail-faq-v1'], 'dense_nnlm': ['travel

## Graded evaluation
These grades belong only to the worked example and never replace KA-0 labels. Empty-gold queries have separate abstention scoring.

In [3]:
print(metrics(['C','A','B'], ['A','B'], k=2, grades={'A':1,'B':2}))
print(metrics(['C','A','B'], ['A','B'], k=3, grades={'A':1,'B':2}))
assert metrics([], [])['recall'] is None


{'recall': 0.5, 'rr': 0.5, 'ndcg': 0.17376534287144002, 'correct_abstention': None}
{'recall': 1.0, 'rr': 0.5, 'ndcg': 0.58688267143572, 'correct_abstention': None}


## Follow every source-to-answer transition
The trace includes parsed line offsets, authored search cards, exact context, proposal and support checks. The same callable generator boundary can accept another implementation, whose model behavior would need separate evaluation.

In [4]:
from rag import run
for locale in ['en', 'zh-hans']:
    question = load('ka4-eval-v1.json')['cases'][0]['question'][locale]
    trace = run(question, locale=locale)
    print(json.dumps(trace, ensure_ascii=False, indent=2))
    assert trace['answer']['amount_yuan'] == 750
    assert trace['validation']['support']['supported']


{
  "producer": "local-extractive-v1; original deterministic Python, not an LLM or replay",
  "question": "Beijing lodging",
  "retrieval_query": "Beijing lodging",
  "on_date": "2026-09-14",
  "topic": "travel",
  "locale": "en",
  "parsed_chunks": [
    {
      "chunk_id": "travel-v1:L1",
      "source_id": "travel-v1",
      "version": "v1",
      "line_start": 1,
      "line_end": 1,
      "start_codepoint": 0,
      "end_codepoint": 65,
      "text": "Fictional Beijing lodging policy; amounts are per room per night.",
      "parent_id": "travel-v1",
      "license": "CC-BY-SA-4.0"
    },
    {
      "chunk_id": "travel-v1:L2",
      "source_id": "travel-v1",
      "version": "v1",
      "line_start": 2,
      "line_end": 2,
      "start_codepoint": 66,
      "end_codepoint": 111,
      "text": "Effective from 2025-01-01 through 2025-12-31.",
      "parent_id": "travel-v1",
      "license": "CC-BY-SA-4.0"
    },
    {
      "chunk_id": "travel-v1:L3",
      "source_id": "travel-v1"

## Paired whole-set evaluation
Only eligibility-before-selection changes. Fault injections remain failures of the required task even if the final gate safely abstains. Three deterministic repeats and translations are not independent semantic cases.

In [5]:
from rag_eval import evaluate, ablations
report = evaluate()
print(json.dumps(report['summaries'], indent=2))
print(json.dumps(report['confidence'], indent=2))
assert len(report['rows']) == 144
assert report['confidence']['paired_differences'] == [0,1,0,0,0,0,0,0,0,0,0,0]
for row in report['rows']:
    if row['locale'] == 'en' and row['repetition'] == 1:
        print(row['case_id'], row['condition'], row['metrics'])


{
  "baseline": {
    "end_to_end": {
      "numerator": 7,
      "denominator": 12,
      "mean": 0.5833333333333334
    },
    "retrieval": {
      "numerator": 8.0,
      "denominator": 10,
      "mean": 0.8
    },
    "context": {
      "numerator": 6.0,
      "denominator": 10,
      "mean": 0.6
    },
    "generation": {
      "numerator": 8,
      "denominator": 12,
      "mean": 0.6666666666666666
    },
    "citation_precision": {
      "numerator": 3.0,
      "denominator": 5,
      "mean": 0.6
    },
    "failures": {
      "passed": 7,
      "retrieval": 2,
      "context": 1,
      "generation": 1,
      "citation": 1
    }
  },
  "filtered": {
    "end_to_end": {
      "numerator": 8,
      "denominator": 12,
      "mean": 0.6666666666666666
    },
    "retrieval": {
      "numerator": 9.0,
      "denominator": 10,
      "mean": 0.9
    },
    "context": {
      "numerator": 7.0,
      "denominator": 10,
      "mean": 0.7
    },
    "generation": {
      "numerator": 9,
 

## Separate ablations
Rewrite, parent expansion and exact-line compression are individual mechanism probes. Their improvements are not folded into the primary filter comparison. The irrelevant filing note is an authored corpus variant.

In [6]:
for row in ablations()['results']:
    trace = row['trace']
    print(row['probe'], row.get('locale', 'en'), row['changed'], trace['context']['input_units'], trace['answer']['status'], trace['answer']['reason_code'])


synonym_rewrite en False 367 abstained insufficient_evidence
synonym_rewrite en True 591 answered None
synonym_rewrite zh-hans False 202 abstained insufficient_evidence
synonym_rewrite zh-hans True 348 answered None
parent_expansion en False 476 abstained insufficient_evidence
parent_expansion en True 591 answered None
exact_line_compression en False 1153 answered None
exact_line_compression en True 591 answered None
